#### Pre-Process for Pictures

In [2]:
import numpy as np
import os
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from tensorflow.keras.utils import image_dataset_from_directory

I0000 00:00:1781422519.824983   33954 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1781422519.828649   33954 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1781422520.112935   33954 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1781422522.201058   33954 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

In [3]:
train_dir = '/home/taha/Desktop/folder/un/cv/final project/dataset/PTSD_Recognition/train'
test_dir = '/home/taha/Desktop/folder/un/cv/final project/dataset/PTSD_Recognition/test'

### This is the label of each picture

In [4]:
test_labels = pd.read_csv(r'/home/taha/Desktop/folder/un/cv/final project/dataset/testset_CSV.csv')
test_labels.head(5)

,file,species
0,00001.jpg,Park
1,00002.jpg,U-Turn Allowed
2,00003.jpg,Compulsory Keep BothSide
3,00004.jpg,One way Traffic
4,00005.jpg,Left Margin


In [5]:
train_data = []
for class_name in os.listdir(train_dir):
    class_path = os.path.join(train_dir,class_name)

    if os.path.isdir(class_path):
        for img in os.listdir(class_path):
            train_data.append([
                os.path.join(class_path, img),
                class_name
            ])

train_df = pd.DataFrame(train_data, columns=["image_path", "label"])

### Image processing pipeline

In [6]:
def pipeline(img_path):
    img = cv2.imread(img_path)
    if img is None:
        raise ValueError(f"Cannot read image: {img_path}")
    # RESIZE ALL PICTURES TO (256,256)
    resized = cv2.resize(img,(255,255))
    #CHANGE COLORING TO YUV FORMAT FOR CNN BETTER UNDERSTANDING
    yuv = cv2.cvtColor(resized,cv2.COLOR_BGR2YUV)
    #ADDING CLAHE FILTER FOR BETTER CONTRAST
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4, 4))
    yuv[:, :, 0] = clahe.apply(yuv[:, :, 0])
    img = cv2.cvtColor(yuv, cv2.COLOR_YUV2BGR)
    #NORMALIZING PICTURE
    img = img.astype(np.float32) / 255.0
    return img

In [7]:
X_test = []
y_test = []
for _,row in test_labels.iterrows():
    image_path = os.path.join(
        'PTSD_'
        test_dir,
        row['file']
    )
    img = pipeline(image_path)
    X_test.append(img)
    y_test.append(row['species'])

X_test = np.array(X_test,dtype=np.float32)
y_test = np.array(y_test)

[ WARN:0@14.718] global loadsave.cpp:278 findDecoder imread_('/home/taha/Desktop/folder/un/cv/final project/dataset/PTSD_Recognition/test/00001.jpg'): can't open/read file: check file path/integrity


ValueError: Cannot read image: /home/taha/Desktop/folder/un/cv/final project/dataset/PTSD_Recognition/test/00001.jpg

In [ ]:
X_train = []
y_train = []

for _,row in train_df.iterrows():
    img = pipeline(row['image_path'])
    X_train.append(img)
    y_train.append(row['label'])

X_train= np.array(X_train, dtype=np.float32)
y_train = np.array(y_train)